# Team Report 1 — March 14, 2026

This week we set up the **Plaid authentication flow** in our FastAPI backend. The goal was to get a working `access_token` for a user's bank connection so we can pull their transaction data. Everything below runs against Plaid's Sandbox environment.

### Notes:

The notebook demos the Plaid auth flow directly, but the overall system requires a FastAPI **backend server** running separately. Plaid sends webhooks (HTTP POST requests) to a public URL when new transactions are available, and those need a persistent server to receive and process them, which notebooks can't do. In local development, we can use ngrok to allow for a public endpoint to serve as the webhook url. The backend also needs to handle secure storage of access tokens, database writes to Supabase, and invoking the AI agent. 

We also need a **minimal frontend** (even just a single HTML page) for two reasons. First, the production Plaid Link flow requires their JavaScript SDK running in a browser. There's no way to complete the real user-facing bank connection without it. Second, the broader application needs some kind of interface for user authentication, displaying agent recommendations, and collecting preference feedback. We think that it would be best in terms of the scope of the project to use the notebook for demo purpopses, while the separate frontend and backend applications serve the real intended purpose.

## Setup

Install dependencies and load Plaid credentials. Locally these come from `backend/.env`.
On Colab, these need to be added to the Secrets panel or enter when prompted.

In [1]:
import sys, os

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    %pip install -q plaid-python python-dotenv

In [2]:
from getpass import getpass

if IS_COLAB:
    try:
        from google.colab import userdata
        PLAID_CLIENT_ID = userdata.get("PLAID_CLIENT_ID")
        PLAID_SECRET = userdata.get("PLAID_SECRET")
        PLAID_ENV = userdata.get("PLAID_ENV") or "sandbox"
    except Exception:
        PLAID_CLIENT_ID = getpass("PLAID_CLIENT_ID: ")
        PLAID_SECRET = getpass("PLAID_SECRET: ")
        PLAID_ENV = input("PLAID_ENV [sandbox]: ") or "sandbox"
else:
    from dotenv import load_dotenv
    load_dotenv(os.path.join(os.getcwd(), "..", "backend", ".env"))
    PLAID_CLIENT_ID = os.environ["PLAID_CLIENT_ID"]
    PLAID_SECRET = os.environ["PLAID_SECRET"]
    PLAID_ENV = os.environ.get("PLAID_ENV", "sandbox")

print(f"Environment: {PLAID_ENV}")

Environment: sandbox


## Initialize the Plaid client

Same configuration our FastAPI backend uses in `backend/app/routers/plaid.py`.

In [3]:
import plaid
from plaid.api import plaid_api

configuration = plaid.Configuration(
    host=plaid.Environment.Sandbox if PLAID_ENV == "sandbox" else plaid.Environment.Production,
    api_key={"clientId": PLAID_CLIENT_ID, "secret": PLAID_SECRET},
)
plaid_client = plaid_api.PlaidApi(plaid.ApiClient(configuration))

## Step 1 — Create a link token

The server creates a short-lived `link_token` via Plaid's API. In production this would be sent to a frontend to open Plaid Link. We only request the `transactions` product since that's all our app needs.

https://plaid.com/docs/api/link/#create-link-token

In [4]:
from plaid.model.link_token_create_request import LinkTokenCreateRequest
from plaid.model.link_token_create_request_user import LinkTokenCreateRequestUser
from plaid.model.products import Products
from plaid.model.country_code import CountryCode

link_response = plaid_client.link_token_create(
    LinkTokenCreateRequest(
        user=LinkTokenCreateRequestUser(client_user_id="test-user-1"), # user id that will be used to identify the user in the database
        client_name="Nutrition and Budgeting Assistant", # name displayed to the user in the Plaid Link UI
        products=[Products("transactions")], # product that we are requesting from the bank
        country_codes=[CountryCode("US")], # country that we are requesting the data from
        language="en", # language that we are requesting the data in
    )
)

print(f"link_token: {link_response.link_token[:40]}...")
print(f"expires:    {link_response.expiration}")

link_token: link-sandbox-252b3a01-c59f-4f7d-84f8-b55...
expires:    2026-03-23 03:13:20+00:00


## Steps 2 & 3 — Simulate user connecting their bank (Sandbox only)

Normally the user would open Plaid Link in a browser, pick their bank, and log in. That flow returns a one-time `public_token`. Since we're in Sandbox, we can generate one directly with `/sandbox/public_token/create` instead.

https://plaid.com/docs/sandbox/

In [5]:
from plaid.model.sandbox_public_token_create_request import SandboxPublicTokenCreateRequest

sandbox_response = plaid_client.sandbox_public_token_create(
    SandboxPublicTokenCreateRequest(
        institution_id="ins_109508",  # Dummy institution for demo purposes
        initial_products=[Products("transactions")], # product that we are requesting, same as in the link token
    )
)
public_token = sandbox_response.public_token
print(f"public_token: {public_token}")

public_token: public-sandbox-2413e7bf-6d3a-4712-9973-a1a39145bb42


## Step 4 — Exchange public token for an access token

The `public_token` is single-use and expires in 30 minutes. We exchange it for a permanent `access_token` (used to pull data) and an `item_id` (identifies this bank connection). In our backend, both get stored in the database.

https://plaid.com/docs/api/items/#exchange-public-token-for-an-access-token

In [6]:
from plaid.model.item_public_token_exchange_request import ItemPublicTokenExchangeRequest

exchange_response = plaid_client.item_public_token_exchange(
    ItemPublicTokenExchangeRequest(public_token=public_token)
)

access_token = exchange_response.access_token
item_id = exchange_response.item_id

print(f"access_token: {access_token}")
print(f"item_id:      {item_id}")

access_token: access-sandbox-7096fc9e-a1c5-4446-8ae7-b126169a3e21
item_id:      Q6P49meBnKHv7DD6md4zha6Ed8NLalHkaEjp7


## Pull sample transactions

With the `access_token` we can now call `/transactions/sync` to fetch transaction data. This uses cursor-based pagination, so we keep calling until `has_more` is false which is based on the latest id pulled.

https://plaid.com/docs/api/products/transactions/#transactionssync

In [7]:
import time
from plaid.model.transactions_sync_request import TransactionsSyncRequest

time.sleep(3)  # give Sandbox a moment to generate test data

cursor = ""
added = []
has_more = True

while has_more:
    response = plaid_client.transactions_sync(
        TransactionsSyncRequest(access_token=access_token, cursor=cursor)
    )
    added.extend(response.added)
    has_more = response.has_more
    cursor = response.next_cursor

print(f"{len(added)} transactions synced\n")

print(f"{'Date':<12} {'Amount':>10}  {'Name':<30} {'Category'}")
print("-" * 78)
for transaction in added[:10]:
    cat = ", ".join(transaction.category) if transaction.category else "—"
    print(f"{str(transaction.date):<12} ${transaction.amount:>9.2f}  {transaction.name[:29]:<30} {cat}")

16 transactions synced

Date             Amount  Name                           Category
------------------------------------------------------------------------------
2026-03-15   $     5.40  Uber 063015 SF**POOL**         —
2026-03-13   $  -500.00  United Airlines                —
2026-03-12   $    12.00  McDonald's                     —
2026-03-12   $     4.33  Starbucks                      —
2026-03-11   $    89.40  SparkFun                       —
2026-02-26   $     6.33  Uber 072515 SF**POOL**         —
2026-03-15   $    25.00  CREDIT CARD 3333 PAYMENT *//   —
2026-03-10   $    -4.22  INTRST PYMNT                   —
2026-03-14   $  1000.00  CD DEPOSIT .INITIAL.           —
2026-03-13   $    78.50  Touchstone Climbing            —


## Cleanup

Remove the Sandbox Item so it doesn't linger between runs.

In [8]:
from plaid.model.item_remove_request import ItemRemoveRequest

plaid_client.item_remove(ItemRemoveRequest(access_token=access_token))
print(f"Item {item_id} removed")

Item Q6P49meBnKHv7DD6md4zha6Ed8NLalHkaEjp7 removed


## Sandbox vs. Production

The code above is identical in both environments. The only differences are configuration values in `.env`:

- **Sandbox:** test credentials, fake bank data, `public_token` can be generated via API shortcut
- **Production:** real Plaid keys, real bank logins via the Plaid Link JS SDK in a browser, `access_token` stored encrypted in DB

No code changes needed — just swap the env vars.

---

## Plan for next week

Next week we'll connect **Supabase** as our database so we can persist data across the application:

- Create tables for `users`, `plaid_items` (stores `access_token` and `item_id`), and `transactions`
- Add a Supabase client to the backend and write the `access_token` + `item_id` there after the exchange step instead of just returning them
- Set up the **webhook endpoint** (`POST /plaid/webhook`) so Plaid can notify us when new transactions are available, and have it trigger `/transactions/sync` to store them in Supabase
- This gives us a persistent data layer that the AI agent can later query when analyzing a user's spending